# Summarization Evaluation: Fine-tuned vs. Pretrained t5-small

Compares the t5-small model fine-tuned on real MTSamples clinical transcriptions
against the plain pretrained t5-small baseline, on two real held-out datasets:

1. **MTSamples test split** (in-domain): real clinical transcription -> real description
2. **PubMed sample** (out-of-domain): real research article -> real abstract

This checks both whether fine-tuning improved in-domain summaries, and whether it
hurt the model's general summarization ability outside the clinical-note domain.

In [ ]:
import sys
sys.path.append('..')

from src.modeling.evaluate_summarizer import (
    evaluate_on_mtsamples,
    evaluate_on_pubmed,
    FINE_TUNED_DIR,
)
from src.modeling.train_summarizer import MODEL_NAME
import pandas as pd

In [ ]:
assert FINE_TUNED_DIR.exists(), "Run `python -m src.modeling.train_summarizer` first"

rows = []
for label, path in [
    ("pretrained-t5-small (baseline)", MODEL_NAME),
    ("fine-tuned-t5-small (ours)", str(FINE_TUNED_DIR)),
]:
    mt = evaluate_on_mtsamples(path)
    pm = evaluate_on_pubmed(path)
    rows.append({"model": label, "dataset": "MTSamples (in-domain)", **mt})
    rows.append({"model": label, "dataset": "PubMed (out-of-domain)", **pm})

results = pd.DataFrame(rows)
results

## Example outputs side by side

A few real MTSamples transcriptions with the human-written reference description,
the pretrained baseline's summary, and the fine-tuned model's summary.

In [ ]:
from src.data_ingestion.load_mtsamples import load_mtsamples
from src.modeling.evaluate_summarizer import summarize_batch

sample = load_mtsamples().sample(5, random_state=123)

baseline_preds = summarize_batch(MODEL_NAME, sample["transcription"].tolist())
finetuned_preds = summarize_batch(str(FINE_TUNED_DIR), sample["transcription"].tolist())

comparison = pd.DataFrame({
    "reference_description": sample["description"].values,
    "pretrained_baseline": baseline_preds,
    "fine_tuned": finetuned_preds,
})
pd.set_option("display.max_colwidth", 120)
comparison